In [ ]:
import json
import pandas as pd


In [3]:
file_path = "../meltano_project/output/ev_batch_sample.json"

In [4]:
with open(file_path, "r") as file:
    data = json.load(file)

In [5]:
import os

os.getcwd()

'/home/bear/GitHub/Project 2/datamall-pipeline/notebook'

In [6]:
data.keys()

dict_keys(['LastUpdatedTime', 'evLocationsData'])

In [7]:
data["LastUpdatedTime"]

'2026-10-02 15:35:00'

In [8]:
len(data["evLocationsData"])

2776

In [9]:
total_charging_points = sum(
    len(location["chargingPoints"])
    for location in data["evLocationsData"]
)

total_charging_points

9109

In [10]:
total_ev_ids = sum(
    len(plug["evIds"])
    for location in data["evLocationsData"]
    for charging_point in location["chargingPoints"]
    for plug in charging_point["plugTypes"]
)

total_ev_ids

11560

In [11]:
ev_ids = [
    ev["evCpId"]
    for location in data["evLocationsData"]
    for charging_point in location["chargingPoints"]
    for plug in charging_point["plugTypes"]
    for ev in plug["evIds"]
]

len(ev_ids)

11560

In [12]:
len(set(ev_ids))

11560

In [13]:
from collections import Counter

status_counts = Counter(
    ev["status"]
    for location in data["evLocationsData"]
    for charging_point in location["chargingPoints"]
    for plug in charging_point["plugTypes"]
    for ev in plug["evIds"]
)

status_counts

Counter({'1': 8835, '0': 2356, '': 369})

In [14]:
data["evLocationsData"][0]

{'address': '142A PASIR RIS STREET 11 SINGAPORE 511142',
 'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
 'longtitude': 103.959758,
 'latitude': 1.362839,
 'postalCode': '511142',
 'chargingPoints': [{'status': '1',
   'operatingHours': '',
   'operator': 'STRIDES YTL PTE. LTD.',
   'position': 'L3A 127',
   'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
   'plugTypes': [{'plugType': 'Type 2',
     'price': '0.6980',
     'current': 'AC',
     'powerRating': '7.4',
     'priceType': 'kWh',
     'evIds': [{'evCpId': 'R122900G-001', 'status': '1'}]}]},
  {'status': '0',
   'operatingHours': '',
   'operator': 'STRIDES YTL PTE. LTD.',
   'position': 'L3A 129',
   'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
   'plugTypes': [{'plugType': 'Type 2',
     'price': '0.6980',
     'current': 'AC',
     'powerRating': '7.4',
     'priceType': 'kWh',
     'evIds': [{'evCpId': 'R122902D-001', 'status': '0'}]}]},
  {'status': '0',
   'operatingHours': 

In [15]:
data["evLocationsData"][0].keys()

dict_keys(['address', 'name', 'longtitude', 'latitude', 'postalCode', 'chargingPoints'])

In [16]:
data["evLocationsData"][0]["chargingPoints"][0].keys()

dict_keys(['status', 'operatingHours', 'operator', 'position', 'name', 'plugTypes'])

In [17]:
data["evLocationsData"][0]

{'address': '142A PASIR RIS STREET 11 SINGAPORE 511142',
 'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
 'longtitude': 103.959758,
 'latitude': 1.362839,
 'postalCode': '511142',
 'chargingPoints': [{'status': '1',
   'operatingHours': '',
   'operator': 'STRIDES YTL PTE. LTD.',
   'position': 'L3A 127',
   'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
   'plugTypes': [{'plugType': 'Type 2',
     'price': '0.6980',
     'current': 'AC',
     'powerRating': '7.4',
     'priceType': 'kWh',
     'evIds': [{'evCpId': 'R122900G-001', 'status': '1'}]}]},
  {'status': '0',
   'operatingHours': '',
   'operator': 'STRIDES YTL PTE. LTD.',
   'position': 'L3A 129',
   'name': 'BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR PARK',
   'plugTypes': [{'plugType': 'Type 2',
     'price': '0.6980',
     'current': 'AC',
     'powerRating': '7.4',
     'priceType': 'kWh',
     'evIds': [{'evCpId': 'R122902D-001', 'status': '0'}]}]},
  {'status': '0',
   'operatingHours': 

In [18]:
data["evLocationsData"][0]["chargingPoints"][0]["plugTypes"][0].keys()

dict_keys(['plugType', 'price', 'current', 'powerRating', 'priceType', 'evIds'])

In [19]:
data["evLocationsData"][0]["chargingPoints"][0]["plugTypes"][0]["evIds"][0].keys()

dict_keys(['evCpId', 'status'])

In [20]:
rows = []

for location in data["evLocationsData"]:
    for charging_point in location["chargingPoints"]:
        for plug in charging_point["plugTypes"]:
            for ev in plug["evIds"]:

                row = {
                    "snapshot_time": data["LastUpdatedTime"],
                    "location_name": location["name"],
                    "address": location["address"],
                    "postal_code": location["postalCode"],
                    "latitude": location["latitude"],
                    "longitude": location["longtitude"],
                    "operator": charging_point["operator"],
                    "position": charging_point["position"],
                    "operating_hours": charging_point["operatingHours"],
                    "charger_status": charging_point["status"],
                    "plug_type": plug["plugType"],
                    "current": plug["current"],
                    "power_rating": plug["powerRating"],
                    "price": plug["price"],
                    "price_type": plug["priceType"],
                    "ev_cp_id": ev["evCpId"],
                    "evcp_status": ev["status"]
                }

                rows.append(row)

In [21]:
len(rows)

11560

In [23]:
df = pd.DataFrame(rows)

df.head()

,snapshot_time,location_name,address,postal_code,latitude,longitude,operator,position,operating_hours,charger_status,plug_type,current,power_rating,price,price_type,ev_cp_id,evcp_status
0,2026-10-02 15:35:00,BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR ...,142A PASIR RIS STREET 11 SINGAPORE 511142,511142,1.362839,103.959758,STRIDES YTL PTE. LTD.,L3A 127,,1,Type 2,AC,7.4,0.6980,kWh,R122900G-001,1
1,2026-10-02 15:35:00,BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR ...,142A PASIR RIS STREET 11 SINGAPORE 511142,511142,1.362839,103.959758,STRIDES YTL PTE. LTD.,L3A 129,,0,Type 2,AC,7.4,0.6980,kWh,R122902D-001,0
2,2026-10-02 15:35:00,BLK 142A PASIR RIS STREET 11 MULTI STOREY CAR ...,142A PASIR RIS STREET 11 SINGAPORE 511142,511142,1.362839,103.959758,STRIDES YTL PTE. LTD.,L3A 128,,0,Type 2,AC,7.4,0.6980,kWh,R122901R-001,0
3,2026-10-02 15:35:00,CLARKE QUAY,3E RIVER VALLEY ROAD SINGAPORE 179024,179024,1.290314,103.846728,SP MOBILITY PTE. LTD.,L4 50,,1,Type 2,AC,22,0.8066,kWh,R114861R-001,1
4,2026-10-02 15:35:00,CLARKE QUAY,3E RIVER VALLEY ROAD SINGAPORE 179024,179024,1.290314,103.846728,SP MOBILITY PTE. LTD.,L4 49,,1,Type 2,AC,22,0.8066,kWh,R114859D-001,1


In [24]:
df.shape

(11560, 17)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11560 entries, 0 to 11559
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   snapshot_time    11560 non-null  object 
 1   location_name    11560 non-null  object 
 2   address          11560 non-null  object 
 3   postal_code      11560 non-null  object 
 4   latitude         11560 non-null  float64
 5   longitude        11560 non-null  float64
 6   operator         11560 non-null  object 
 7   position         11560 non-null  object 
 8   operating_hours  11560 non-null  object 
 9   charger_status   11560 non-null  object 
 10  plug_type        11560 non-null  object 
 11  current          11560 non-null  object 
 12  power_rating     11560 non-null  object 
 13  price            11560 non-null  object 
 14  price_type       11560 non-null  object 
 15  ev_cp_id         11560 non-null  object 
 16  evcp_status      11560 non-null  object 
dtypes: float64(2